# Game of 24 — Tree of Thoughts (GPT-4o mini)
**CS 4782 Final Project — Cornell University**

Replicates Table 2 from Yao et al. NeurIPS 2023 using **GPT-4o mini** via the OpenAI API.
Estimated cost: **~$1–2** for all methods on 100 puzzles.

Paper numbers (GPT-4): IO=7.3%, CoT=4%, CoT-SC=9%, ToT-BFS=74%.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q openai datasets matplotlib numpy

## Cell 2 — Set your OpenAI API key
Get a key at **platform.openai.com** → API Keys → Create new secret key.
You need at least **$5 credit** loaded (Settings → Billing).

In [ ]:
import os

OPENAI_API_KEY = "sk-YOUR_KEY_HERE"   # ← paste your OpenAI key

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("Key set." if OPENAI_API_KEY != "sk-YOUR_KEY_HERE" else "ERROR: replace the placeholder key above!")

## Cell 3 — Clone repo

In [ ]:
import sys, os, subprocess

os.chdir("/content")
repo = "/content/Tree-of-Thoughts-Debugger"
if not os.path.exists(repo):
    subprocess.run(["git", "clone", "https://github.com/Datboy0127/Tree-of-Thoughts-Debugger.git"],
                   cwd="/content", check=True)
    print("Cloned.")
else:
    out = subprocess.run(["git", "pull"], cwd=repo, capture_output=True, text=True)
    print("Pulled:", out.stdout.strip() or "up to date")

sys.path.insert(0, f"{repo}/code")
os.chdir(repo)
print("Ready. CWD:", os.getcwd())

## Cell 4 — (Optional) Download paper's exact puzzle set
Problems 901-1000 from 4nums.com. Skip to use built-in hard puzzles.

In [ ]:
import urllib.request, os

os.makedirs("data", exist_ok=True)
csv_url = "https://raw.githubusercontent.com/princeton-nlp/tree-of-thought-llm/master/src/tot/data/24/24.csv"
try:
    urllib.request.urlretrieve(csv_url, "data/24.csv")
    print("Downloaded data/24.csv")
    GAME24_CSV = "data/24.csv"
except Exception as e:
    print(f"Download failed ({e}) — using built-in puzzle list.")
    GAME24_CSV = None

## Cell 5 — Load puzzles

In [ ]:
from data_loader import load_game24

G24_N = 100   # set to 20 for a quick test

puzzles = load_game24(n=G24_N, csv_path=GAME24_CSV, seed=42)
print(f"Loaded {len(puzzles)} puzzles")
print("First 5:", puzzles[:5])

## Cell 6 — Setup LLM (GPT-4o mini)
Uses OpenAI API. Estimated cost: ~$1–2 for all 6 methods on 100 puzzles.

In [ ]:
import config

config.BACKEND = "openai"
config.MODEL   = "gpt-4o-mini"

from llm_client import LLMClient
from game24_solver import (
    Game24Solver, Game24IOBaseline, Game24CoTBaseline, Game24CoTSCBaseline,
    Game24MCTSSolver, run_game24, compute_game24_metrics, print_game24_table,
)

llm = LLMClient()
print(f"Model  : {llm.model}")
print(f"Backend: {llm.backend}")
print(f"URL    : {llm.base_url}")

# Quick connectivity test
try:
    resp = llm.call("Say 'ok' in one word.", max_tokens=5)
    print(f"Connection OK — model replied: {resp['content'].strip()}")
except Exception as e:
    print(f"ERROR: {e}")
    print("Check your API key in Cell 2 and that you have billing set up.")

import os
os.makedirs("results", exist_ok=True)
g24 = {}

## Cell 7 — IO baseline
Paper (GPT-4): 7.3%

In [ ]:
import json

print("── IO ──")
g24["io"] = run_game24(puzzles, Game24IOBaseline(llm), "io", verbose=True)
m = compute_game24_metrics(g24["io"])
print(f"\n  IO: {m['success_rate']:.1%}  (paper GPT-4: 7.3%)")

with open("results/game24_io.json", "w") as f:
    json.dump([r.__dict__ for r in g24["io"]], f, indent=2)
print("Saved results/game24_io.json")

## Cell 8 — CoT baseline
Paper (GPT-4): 4%

In [ ]:
import json

print("── CoT ──")
g24["cot"] = run_game24(puzzles, Game24CoTBaseline(llm), "cot", verbose=True)
m = compute_game24_metrics(g24["cot"])
print(f"\n  CoT: {m['success_rate']:.1%}  (paper GPT-4: 4%)")

with open("results/game24_cot.json", "w") as f:
    json.dump([r.__dict__ for r in g24["cot"]], f, indent=2)
print("Saved results/game24_cot.json")

## Cell 9 — CoT-SC (n=5)
Paper uses n=100 for 9%; we use n=5 for cost efficiency.

In [ ]:
import json

print("── CoT-SC (n=5) ──")
g24["cot-sc-5"] = run_game24(puzzles, Game24CoTSCBaseline(llm, n_samples=5), "cot-sc-5", verbose=True)
m = compute_game24_metrics(g24["cot-sc-5"])
print(f"\n  CoT-SC (n=5): {m['success_rate']:.1%}  (paper GPT-4 n=100: 9%)")

with open("results/game24_cot-sc-5.json", "w") as f:
    json.dump([r.__dict__ for r in g24["cot-sc-5"]], f, indent=2)
print("Saved results/game24_cot-sc-5.json")

## Cell 10 — ToT-BFS (b=5)
Main paper result: **74%** on GPT-4.
Algorithm 1: propose → value → keep top-b at each of 3 depth levels.

In [ ]:
import json

print("── ToT-BFS (b=5) ──")
g24["tot-bfs-b5"] = run_game24(puzzles, Game24Solver(llm, b=5, search="bfs"), "tot-bfs-b5", verbose=True)
m = compute_game24_metrics(g24["tot-bfs-b5"])
print(f"\n  ToT-BFS: {m['success_rate']:.1%}  avg_nodes={m['avg_nodes']:.1f}  avg_time={m['avg_time']:.1f}s")
print(f"  Paper (GPT-4): 74%")

with open("results/game24_tot-bfs-b5.json", "w") as f:
    json.dump([r.__dict__ for r in g24["tot-bfs-b5"]], f, indent=2)
print("Saved results/game24_tot-bfs-b5.json")

## Cell 11 — ToT-DFS
Algorithm 2: greedy descent, prune impossible states, backtrack on dead ends.

In [ ]:
import json

print("── ToT-DFS ──")
g24["tot-dfs-b5"] = run_game24(puzzles, Game24Solver(llm, b=5, search="dfs"), "tot-dfs-b5", verbose=True)
m = compute_game24_metrics(g24["tot-dfs-b5"])
print(f"\n  ToT-DFS: {m['success_rate']:.1%}  avg_nodes={m['avg_nodes']:.1f}")

with open("results/game24_tot-dfs-b5.json", "w") as f:
    json.dump([r.__dict__ for r in g24["tot-dfs-b5"]], f, indent=2)
print("Saved results/game24_tot-dfs-b5.json")

## Cell 12 — MCTS (novel extension)
UCB1 over arithmetic states. Rollouts are pure math — no extra API calls.
Early stopping: halts as soon as any simulation finds 24.

In [ ]:
import json

print("── MCTS (b=5, s=20) ──")
g24["mcts-b5-s20"] = run_game24(puzzles, Game24MCTSSolver(llm, b=5, n_simulations=20), "mcts-b5-s20", verbose=True)
m = compute_game24_metrics(g24["mcts-b5-s20"])
print(f"\n  MCTS: {m['success_rate']:.1%}  avg_nodes={m['avg_nodes']:.1f}")

with open("results/game24_mcts-b5-s20.json", "w") as f:
    json.dump([r.__dict__ for r in g24["mcts-b5-s20"]], f, indent=2)
print("Saved results/game24_mcts-b5-s20.json")

## Cell 13 — Results table

In [ ]:
import json

print_game24_table(g24)

# Save summary
summary = {label: compute_game24_metrics(results) for label, results in g24.items()}
with open("results/game24_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\nSaved results/game24_summary.json")

## Cell 14 — Comparison plot: ours vs paper (GPT-4)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

paper   = {"IO": 7.3, "CoT": 4.0, "CoT-SC\n(n=100)": 9.0, "ToT-BFS\n(b=5)": 74.0}
key_map = {"IO": "io", "CoT": "cot", "CoT-SC\n(n=100)": "cot-sc-5", "ToT-BFS\n(b=5)": "tot-bfs-b5"}
ours    = {k: compute_game24_metrics(g24.get(key_map[k], [])).get("success_rate", 0) * 100 for k in paper}

labels = list(paper.keys())
x, w = np.arange(len(labels)), 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, [paper[l] for l in labels], w, label="Paper (GPT-4)",        color="#2196F3", alpha=0.85)
b2 = ax.bar(x + w/2, [ours[l]  for l in labels], w, label="Ours (GPT-4o mini)",   color="#4CAF50", alpha=0.85)
ax.bar_label(b1, fmt="%.1f%%", padding=3, fontsize=9)
ax.bar_label(b2, fmt="%.1f%%", padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("Success Rate (%)", fontsize=12)
ax.set_title("Game of 24 — Paper (GPT-4) vs Ours (GPT-4o mini)", fontsize=13)
ax.set_ylim(0, 90); ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.savefig("results/game24_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/game24_comparison.png")

## Cell 15 — Download results

In [ ]:
import shutil
shutil.make_archive("/content/game24_results", "zip", "results")
print("Zipped → /content/game24_results.zip")
try:
    from google.colab import files
    files.download("/content/game24_results.zip")
except ImportError:
    pass